# Vision third encoder: supervised ViT on CIFAR-10 and Fashion-MNIST (N=1000)

Completes the vision block to three encoders spanning three pretraining
paradigms: contrastive (CLIP), self-supervised (DINOv2), supervised (ViT).
Encoder: `google/vit-base-patch16-224`, CLS token, L2 normalized.

Protocol identical to the 26 existing configurations. Runs both datasets in
sequence with per-dataset checkpointing; a disconnect never costs more than
one kernel matrix (~85 min each, ~3.5 h total).

In [ ]:
!pip install pennylane pennylane-lightning transformers datasets torchvision --quiet

import pennylane as qml
import numpy as np
import torch, time, json, pickle, os, shutil, warnings
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import rbf_kernel
warnings.filterwarnings('ignore')
print(f"PennyLane {qml.__version__} | torch {torch.__version__}")
DEVICE = qml.device('lightning.qubit', wires=8)
SEED = 42

---
## Step 0: Drive storage

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)

def _persist(path):
    dst = os.path.join(SAVE_DIR, os.path.basename(path))
    if os.path.abspath(path) != os.path.abspath(dst):
        shutil.copy(path, dst)
    print(f"  saved -> {dst}")

def checkpoint(filename, save_fn):
    save_fn(filename); _persist(filename)

def find_file(fname):
    dp = os.path.join(SAVE_DIR, fname)
    return dp if os.path.exists(dp) else (fname if os.path.exists(fname) else None)
print("ready")

---
## Step 1: Datasets (identical classes and sampling to existing vision configs)

In [ ]:
N_TRAIN, N_TEST = 1000, 200

def balanced_indices(labels, class_a, class_b, n_total, seed):
    labels = np.asarray(labels)
    rng = np.random.RandomState(seed)
    idx_a = np.where(labels == class_a)[0]; rng.shuffle(idx_a)
    idx_b = np.where(labels == class_b)[0]; rng.shuffle(idx_b)
    half = n_total // 2
    idx = np.concatenate([idx_a[:half], idx_b[:half]])
    rng.shuffle(idx)
    return idx

def load_cifar():
    try:
        from datasets import load_dataset
        ds = load_dataset('uoft-cs/cifar10')
        col = 'img' if 'img' in ds['train'].column_names else 'image'
        ti = balanced_indices(ds['train']['label'], 0, 1, N_TRAIN, SEED)
        si = balanced_indices(ds['test']['label'],  0, 1, N_TEST,  SEED)
        tr, te = ds['train'].select(ti), ds['test'].select(si)
        return tr[col], np.array(tr['label']), te[col], np.array(te['label'])
    except Exception as e:
        print(f"datasets failed ({type(e).__name__}); torchvision fallback")
        from torchvision.datasets import CIFAR10
        tr_ds = CIFAR10(root='./c', train=True, download=True)
        te_ds = CIFAR10(root='./c', train=False, download=True)
        ti = balanced_indices(tr_ds.targets, 0, 1, N_TRAIN, SEED)
        si = balanced_indices(te_ds.targets, 0, 1, N_TEST, SEED)
        return ([tr_ds[int(i)][0] for i in ti], np.array([tr_ds.targets[int(i)] for i in ti]),
                [te_ds[int(i)][0] for i in si], np.array([te_ds.targets[int(i)] for i in si]))

def load_fashion():
    # T-shirt/top (0) vs Trouser (1), as in fashion_clip / fashion_dinov2
    try:
        from datasets import load_dataset
        ds = load_dataset('zalando-datasets/fashion_mnist')
        col = 'image' if 'image' in ds['train'].column_names else 'img'
        ti = balanced_indices(ds['train']['label'], 0, 1, N_TRAIN, SEED)
        si = balanced_indices(ds['test']['label'],  0, 1, N_TEST,  SEED)
        tr, te = ds['train'].select(ti), ds['test'].select(si)
        return tr[col], np.array(tr['label']), te[col], np.array(te['label'])
    except Exception as e:
        print(f"datasets failed ({type(e).__name__}); torchvision fallback")
        from torchvision.datasets import FashionMNIST
        tr_ds = FashionMNIST(root='./f', train=True, download=True)
        te_ds = FashionMNIST(root='./f', train=False, download=True)
        ti = balanced_indices(tr_ds.targets.numpy(), 0, 1, N_TRAIN, SEED)
        si = balanced_indices(te_ds.targets.numpy(), 0, 1, N_TEST, SEED)
        return ([tr_ds[int(i)][0] for i in ti], np.array([int(tr_ds.targets[i]) for i in ti]),
                [te_ds[int(i)][0] for i in si], np.array([int(te_ds.targets[i]) for i in si]))

DATASETS = {'cifar10_vit': load_cifar, 'fashion_vit': load_fashion}
print("loaders defined:", list(DATASETS))

---
## Step 2: ViT encoder (CLS token, L2 normalized)

In [ ]:
from transformers import AutoImageProcessor, AutoModel
proc  = AutoImageProcessor.from_pretrained('google/vit-base-patch16-224')
model = AutoModel.from_pretrained('google/vit-base-patch16-224').eval()
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(dev)
print(f"ViT-base on {dev}")

@torch.no_grad()
def embed(images, batch=64):
    out = []
    for i in range(0, len(images), batch):
        chunk = [im.convert('RGB') for im in images[i:i+batch]]
        inputs = proc(images=chunk, return_tensors='pt').to(dev)
        h = model(**inputs).last_hidden_state[:, 0, :]      # CLS token
        h = h / torch.linalg.norm(h, dim=1, keepdim=True)   # L2 normalize
        out.append(h.cpu().numpy())
    return np.vstack(out)
print("embed() ready (768-dim)")

---
## Step 3: Shared pipeline pieces (identical definitions to the grid)

In [ ]:
def bf1(yt, yp): return float(f1_score(yt, yp, average='binary'))

def iqp_feature_map(x, wires):
    n = len(wires)
    for w in wires: qml.Hadamard(wires=w)
    for i, w in enumerate(wires): qml.RZ(x[i], wires=w)
    for i in range(n - 1):
        qml.CNOT(wires=[wires[i], wires[i+1]])
        qml.RZ(x[i] * x[i+1], wires=wires[i+1])
        qml.CNOT(wires=[wires[i], wires[i+1]])

@qml.qnode(DEVICE)
def kernel_circuit(x1, x2):
    iqp_feature_map(x1, wires=range(8))
    qml.adjoint(iqp_feature_map)(x2, wires=range(8))
    return qml.probs(wires=range(8))

quantum_kernel = lambda a, b: float(kernel_circuit(a, b)[0])

def kta(K, y):
    y = np.asarray(y); n = len(y)
    Y = np.zeros((n, n))
    for c in np.unique(y):
        m = (y == c).astype(float); Y += np.outer(m, m)
    H = np.eye(n) - np.ones((n, n))/n
    Kc, Yc = H @ K @ H, H @ Y @ H
    return float(np.sum(Kc*Yc) / np.sqrt(np.sum(Kc*Kc) * np.sum(Yc*Yc)))

def concentration(K):
    mu = np.linalg.eigvalsh(K); mu = np.maximum(mu, 0)
    lam = mu / mu.sum()
    return float(np.exp(-np.sum(lam * np.log(lam + 1e-15))) / K.shape[0])

def geometric_difference_exact(K_Q, K_C, lam=1e-6):
    n = K_Q.shape[0]
    K_Q = K_Q * n / np.trace(K_Q); K_C = K_C * n / np.trace(K_C)
    w, V = np.linalg.eigh(K_Q); w = np.maximum(w, 0)
    sq = (V * np.sqrt(w)) @ V.T
    return float(np.sqrt(max(np.linalg.eigvalsh(sq @ np.linalg.inv(K_C + lam*n*np.eye(n)) @ sq).max(), 0.0)))

def geometric_difference_proxy(K_Q, K_C):
    eff = lambda K: (np.trace(K)**2) / (np.sum(K**2) + 1e-10)
    return float(eff(K_Q) / (eff(K_C) + 1e-10))

assert np.isclose(quantum_kernel(np.linspace(0,3,8), np.linspace(0,3,8)), 1.0, atol=1e-5)
print("pipeline pieces verified")

---
## Step 4: Run both configurations

Each dataset: embed -> PCA-8 -> [0, pi] -> baselines -> kernel (~85 min) ->
QSVM -> diagnostics -> bootstrap -> record. Checkpoints after every stage;
re-running skips anything already on Drive.

In [ ]:
for TAG, loader in DATASETS.items():
    print(f"\n{'='*60}\n{TAG}\n{'='*60}")
    rec_f = f'{TAG}_grid_record.json'
    if find_file(rec_f):
        print("record exists on Drive; skipping"); continue

    # ---- data + embeddings (checkpointed) ----
    emb_f = f'{TAG}_embeddings.npz'
    hit = find_file(emb_f)
    if hit:
        z = np.load(hit)
        X_tr, y_tr, X_te, y_te = z['X_tr_subset'], z['y_tr_subset'], z['X_te_subset'], z['y_te_subset']
        print("embeddings loaded from checkpoint")
    else:
        tr_imgs, y_tr, te_imgs, y_te = loader()
        print(f"train {len(tr_imgs)}, test {len(te_imgs)}, balance {np.bincount(y_tr)}")
        t0 = time.time()
        E_tr, E_te = embed(tr_imgs), embed(te_imgs)
        print(f"embedded in {(time.time()-t0)/60:.1f} min")
        pca = PCA(n_components=8, random_state=SEED).fit(E_tr)
        P_tr, P_te = pca.transform(E_tr), pca.transform(E_te)
        print(f"PCA-8 variance: {pca.explained_variance_ratio_.sum():.4f}")
        scaler = MinMaxScaler(feature_range=(0, np.pi)).fit(P_tr)
        X_tr = np.clip(scaler.transform(P_tr), 0, np.pi)
        X_te = np.clip(scaler.transform(P_te), 0, np.pi)
        checkpoint(emb_f, lambda p: np.savez(p, X_tr_subset=X_tr, y_tr_subset=y_tr,
                                             X_te_subset=X_te, y_te_subset=y_te))
    assert X_tr.shape == (1000, 8) and X_te.shape == (200, 8)
    assert X_tr.min() >= -1e-9 and X_tr.max() <= np.pi + 1e-9

    # ---- classical baselines ----
    baselines = {}
    clfs = {'Linear': SVC(kernel='linear', C=1.0, random_state=SEED),
            'RBF': SVC(kernel='rbf', C=1.0, gamma='scale', random_state=SEED),
            'Polynomial': SVC(kernel='poly', degree=3, coef0=1, C=1.0, gamma='scale', random_state=SEED)}
    fitted = {}
    for name, clf in clfs.items():
        yp = clf.fit(X_tr, y_tr).predict(X_te)
        fitted[name] = clf
        baselines[name] = {'accuracy': float(accuracy_score(y_te, yp)), 'f1': bf1(y_te, yp)}
        print(f"  {name:<12} f1={baselines[name]['f1']:.4f}")
    best_nm = max(baselines, key=lambda k: baselines[k]['f1'])
    best_f1 = baselines[best_nm]['f1']

    # ---- quantum kernel (checkpointed; the long step) ----
    ker_f = f'{TAG}_kernel_matrices.npz'
    hit = find_file(ker_f)
    if hit:
        z = np.load(hit); K_train, K_test = z['K_train_quantum'], z['K_test_quantum']
        elapsed = float(z['elapsed_min']) if 'elapsed_min' in z.files else -1
        print("kernel loaded from checkpoint")
    else:
        print("computing kernel (~85 min)...")
        t0 = time.time()
        K_train = qml.kernels.kernel_matrix(X_tr, X_tr, quantum_kernel)
        K_test  = qml.kernels.kernel_matrix(X_te, X_tr, quantum_kernel)
        elapsed = (time.time()-t0)/60
        print(f"done in {elapsed:.1f} min")
        checkpoint(ker_f, lambda p: np.savez(p, K_train_quantum=K_train, K_test_quantum=K_test,
                                             X_tr=X_tr, y_tr=y_tr, X_te=X_te, y_te=y_te,
                                             elapsed_min=elapsed))

    # ---- QSVM + gap ----
    qsvm = SVC(kernel='precomputed', C=1.0).fit(K_train, y_tr)
    y_pred = qsvm.predict(K_test)
    f1_q = bf1(y_te, y_pred); acc_q = float(accuracy_score(y_te, y_pred)); gap = f1_q - best_f1
    print(f"quantum F1={f1_q:.4f} | best classical {best_f1:.4f} ({best_nm}) | gap {gap:+.4f}")

    # ---- diagnostics ----
    Krbf = rbf_kernel(X_tr, X_tr, gamma=1.0/X_tr.shape[1])
    A_Q, A_C = kta(K_train, y_tr), kta(Krbf, y_tr)
    rho_, g_ex, g_px = concentration(K_train), geometric_difference_exact(K_train, Krbf), geometric_difference_proxy(K_train, Krbf)
    off_q = K_train[~np.eye(1000, dtype=bool)].mean(); off_c = Krbf[~np.eye(1000, dtype=bool)].mean()
    lam_sweep = {f"{l:.0e}": round(geometric_difference_exact(K_train, Krbf, l), 2)
                 for l in [1e-4, 1e-5, 1e-6, 1e-7]}
    print(f"A_Q={A_Q:.4f} A_C={A_C:.4f} rho={rho_:.4f} g={g_ex:.2f} (proxy {g_px:.2f})")
    print(f"lambda sweep: {lam_sweep}")

    # ---- bootstrap CI ----
    rng = np.random.RandomState(SEED)
    yc_pred = fitted[best_nm].predict(X_te)
    boot = []
    for _ in range(1000):
        idx = rng.randint(0, len(y_te), len(y_te))
        if len(np.unique(y_te[idx])) < 2: continue
        boot.append(bf1(y_te[idx], y_pred[idx]) - bf1(y_te[idx], yc_pred[idx]))
    ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
    print(f"gap CI [{ci_lo:+.4f}, {ci_hi:+.4f}]")

    # ---- learning curve (reuses kernel) ----
    sizes = [50, 100, 200, 400, 700, 1000]
    lc = {'Quantum (IQP)': [], 'Linear': [], 'RBF': []}
    for nn in sizes:
        yp = SVC(kernel='precomputed', C=1.0).fit(K_train[:nn,:nn], y_tr[:nn]).predict(K_test[:,:nn])
        lc['Quantum (IQP)'].append(bf1(y_te, yp))
        lc['Linear'].append(bf1(y_te, SVC(kernel='linear', C=1.0, random_state=SEED).fit(X_tr[:nn], y_tr[:nn]).predict(X_te)))
        lc['RBF'].append(bf1(y_te, SVC(kernel='rbf', C=1.0, gamma='scale', random_state=SEED).fit(X_tr[:nn], y_tr[:nn]).predict(X_te)))

    record = {'tag': TAG, 'domain': 'vision', 'encoder': 'vit',
              'dataset': 'cifar10_binary' if 'cifar' in TAG else 'fashion_binary',
              'metric': 'binary', 'n_qubits': 8, 'n_train': 1000, 'n_test': 200,
              'compute_time_min': round(elapsed, 1),
              'quantum_accuracy': acc_q, 'quantum_f1': f1_q, 'best_classical_f1': best_f1, 'best_classical_kernel': best_nm,
              'gap': gap, 'kta': A_Q, 'c_kta': A_C, 'concentration': rho_,
              'geometric_difference_exact': g_ex, 'geometric_difference_proxy': g_px, 'exact_g_lambda_sweep': lam_sweep,
              'offdiag_quantum': float(off_q), 'offdiag_rbf': float(off_c),
              'ci_lo': float(ci_lo), 'ci_hi': float(ci_hi),
              'excludes_zero': bool(ci_lo > 0 or ci_hi < 0),
              'classical_baselines': baselines,
              'learning_curve_sizes': sizes, 'learning_curve_results': lc}
    checkpoint(rec_f, lambda p: json.dump(record, open(p, 'w'), indent=2))

print("\nAll done. Upload cifar10_vit_grid_record.json and fashion_vit_grid_record.json.")